# 接触对象：成环尺度谱 P(|i - j|) 与长程自闭压缩分析

本 Notebook 将原本被单一标量均值（约 3.56 contacts/residue）掩盖的分子内拓扑差异进行深度解耦：
1. **成环尺度分布谱 $P(|i - j|)$**：对标分子间伙伴分布，展示短程发卡与长程大闭环的尺度竞争；
2. **长程大闭环（$|i-j| > 30$）累积占比**：定量揭示极端 SCD 变体（如 B10）的“首尾相吸自锁”倾向；
3. **长程闭环对单链尺寸 $R_g$ 的几何压缩效应**：证明因果关系——长程大闭环直接导致链构象坍塌，从而失去与外界形成多聚相分离网络的能力。

在首个代码单元选择 OBJECT：`dense_chains` 为全部浓相链（推荐），`central_chain` 为每帧几何中心链。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulations.yaml").is_file())
sys.path.insert(0, str(ROOT))
from common import inputs
from common import results as files

SELECTED = None  # None 为全部已计算模拟
OBJECT = "dense_chains"  # dense_chains / central_chain
RESULTS_ROOT = ROOT / "03_results"
SIMULATIONS = inputs.select_simulations(ROOT / "simulations.yaml", SELECTED)
RESULTS = {
    s["id"]: files.completed(RESULTS_ROOT / s["id"] / "02_contacts/01_intra" / OBJECT)
    for s in SIMULATIONS
    if (RESULTS_ROOT / s["id"] / "02_contacts/01_intra" / OBJECT / "COMPLETE").is_file()
}
SOURCES = {name: files.read_json(path / "metadata.json") for name, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = ROOT / "05_plot/02_contacts/01_intra/figures" / OBJECT
FIGURES.mkdir(parents=True, exist_ok=True)

# Nature Standard Style
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8.5,
    "axes.titlesize": 9,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "svg.fonttype": "none",
})

def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart notebook")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    files.write_json(FIGURES / f"{name}.sources.json", SOURCES)


In [ ]:
# 读取已提取的成环尺度谱与逐链记录
P_LOOPS = {}
CHAIN_DFS = {}
for name, path in RESULTS.items():
    p_file = path / "loop_size_distribution.npy"
    if p_file.is_file():
        P_LOOPS[name] = np.load(p_file)
    c_file = path / "chain_contacts.csv.gz"
    if c_file.is_file():
        df = pd.read_csv(c_file)
        if "long_range_contacts" in df.columns:
            CHAIN_DFS[name] = df

print(f"Loaded loop distributions for: {list(P_LOOPS.keys())}")


### 图 1：成环尺度谱 P(|i - j|) 与长程大闭环累积分布

In [ ]:
if P_LOOPS:
    fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(7.2, 2.7), constrained_layout=True)

    # 1A: 对数尺度下的成环尺度谱 P(|i - j|)
    k_range = np.arange(129)
    for name, p_loop in P_LOOPS.items():
        ax1a.plot(k_range[3:], p_loop[3:], label=name, color=COLORS[name], lw=1.2)

    ax1a.set_yscale("log")
    ax1a.set_xlabel("Sequence separation $|i - j|$ (residues)")
    ax1a.set_ylabel("Contact probability $P(|i - j|)$")
    ax1a.set_title("a | Loop Size Distribution", loc="left", fontweight="bold")
    ax1a.set_xlim(3, 125)
    ax1a.legend(loc="upper right")

    # 1B: 长程大闭环累积概率
    cutoffs = [20, 40, 60, 80]
    x = np.arange(len(cutoffs))
    w = 0.8 / len(P_LOOPS)
    for i, (name, p_loop) in enumerate(P_LOOPS.items()):
        vals = [p_loop[c:].sum() * 100 for c in cutoffs]
        ax1b.bar(x + (i - len(P_LOOPS)/2 + 0.5) * w, vals, width=w, label=name, color=COLORS[name], edgecolor="none")

    ax1b.set_xticks(x)
    ax1b.set_xticklabels([f"$\\geq {c}$" for c in cutoffs])
    ax1b.set_xlabel("Loop size threshold (residues)")
    ax1b.set_ylabel("Cumulative loop fraction (%)")
    ax1b.set_title("b | Long-range Self-trapping Loops", loc="left", fontweight="bold")
    ax1b.legend(loc="upper right")

    export(fig1, "loop_size_distribution")
    plt.show()


### 图 2：长程闭环与单链 Rg 的几何压缩坍塌效应

In [ ]:
if CHAIN_DFS:
    fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(7.2, 2.7), constrained_layout=True)

    for name, df in CHAIN_DFS.items():
        mean_long = df.groupby("contact_count")["long_range_contacts"].mean().reset_index()
        mean_long = mean_long[mean_long["contact_count"].between(30, 100)]
        ax2a.plot(mean_long["contact_count"], mean_long["long_range_contacts"], label=name, color=COLORS[name], lw=1.2)

    ax2a.set_xlabel("Total intrachain contacts ($E_{\\mathrm{intra}}$)")
    ax2a.set_ylabel("Long-range contacts ($|i-j| > 30$)")
    ax2a.set_title("a | Long-range Contact Scaling", loc="left", fontweight="bold")
    ax2a.legend(loc="upper left")

    for name, df in CHAIN_DFS.items():
        rg_by_long = df.groupby("long_range_contacts")["rg_nm"].agg(["mean", "sem"]).reset_index()
        rg_by_long = rg_by_long[rg_by_long["long_range_contacts"].between(0, 16)]
        x_val = rg_by_long["long_range_contacts"].to_numpy(float)
        y_val = rg_by_long["mean"].to_numpy(float)
        y_err = rg_by_long["sem"].to_numpy(float)
        ax2b.plot(x_val, y_val, marker="o", markersize=3, label=name, color=COLORS[name], lw=1.1)
        ax2b.fill_between(x_val, y_val - y_err, y_val + y_err, color=COLORS[sid if 'sid' in locals() else name], alpha=0.2)

    ax2b.set_xlabel("Number of long-range contacts ($|i-j| > 30$)")
    ax2b.set_ylabel("Radius of gyration $R_g$ (nm)")
    ax2b.set_title("b | Chain Compaction by Long-range Loops", loc="left", fontweight="bold")
    ax2b.legend(loc="upper right")

    export(fig2, "long_range_vs_rg_compaction")
    plt.show()
